# Migración Clínica X → Flowww

## Tabla clients

He quitado el campo `saldounidad` de `ClientCustom1`.

Mantengo también a los empleados, porque además son clientes de la clínica y tienen tratamientos, presupuestos, etc.

El `ClinicID` se toma del archivo de origen (`df_origen["ClinicID"]`). Si ese campo es nulo, está vacío o vale 0, se le asigna el valor por defecto 1 mediante la función `aplicar_default_clinicid`.

In [ ]:
import pandas as pd
import os
import re
from decimal import Decimal, InvalidOperation
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# =========================================================
# CONFIG
# =========================================================
RUTA_PLANTILLA = r"C:\Migracion\ClinicaX\Origen\clients.csv"
RUTA_SALIDA    = r"C:\Migracion\ClinicaX\Destino\clients.csv"

CLINICID_DEFAULT = "1"

# =========================================================
# UTILIDADES
# =========================================================
def convertir_fecha(valor) -> str:
    if valor is None or pd.isna(valor):
        return ""
    try:
        return pd.to_datetime(valor).strftime("%d/%m/%Y")
    except Exception:
        return ""

def limpiar_entero_como_texto(x) -> str:
    if x is None or pd.isna(x):
        return ""
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s

def limpiar_telefono(v) -> str:
    if v is None or pd.isna(v):
        return ""

    if isinstance(v, (int, float)):
        try:
            return str(int(v))
        except Exception:
            return ""

    s = str(v).strip()

    if "E+" in s.upper() or "E-" in s.upper():
        try:
            s_norm = s.replace(",", ".")
            return str(int(Decimal(s_norm)))
        except (InvalidOperation, ValueError):
            return ""

    numeros = re.findall(r"\d+", s)
    return "".join(numeros)

def normalizar_sexo(v) -> str:
    if v is None or pd.isna(v):
        return "F"

    s = str(v).strip().upper()
    if s in ["", "NULL", "NONE", "N/A"]:
        return "F"

    if s in ["M", "F"]:
        return s

    if s in ["H", "HOMBRE"]:
        return "M"

    if s in ["V", "MUJER"]:
        return "F"

    return "F"

def aplicar_default_clinicid(v, default_value: str) -> str:
    if v is None or pd.isna(v):
        return default_value

    if isinstance(v, (int, float)):
        try:
            return default_value if float(v) == 0 else str(int(v))
        except Exception:
            return default_value

    s = str(v).strip()
    if s == "":
        return default_value

    su = s.upper()
    if su in ["NULL", "NONE", "N/A"]:
        return default_value

    s_clean = s[:-2] if s.endswith(".0") else s
    if s_clean == "0":
        return default_value

    return s_clean

def normalizar_cp(v) -> str:
    if v is None or pd.isna(v):
        return ""

    s = str(v).strip()
    if s.endswith(".0"):
        s = s[:-2]

    digits = "".join(re.findall(r"\d+", s))
    if digits == "":
        return ""

    if len(digits) > 5:
        digits = digits[-5:]

    if len(digits) < 5:
        digits = digits.zfill(5)

    return digits

# =========================================================
# ENV
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

# =========================================================
# 1) CONEXIÓN SQL SERVER (SQLAlchemy + Windows Auth)
# =========================================================
print("🔌 Conectando a SQL Server...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("✅ Conexión OK")

# =========================================================
# 2) ORIGEN: VISTA SQL dbo.clients
# =========================================================
df_origen = pd.read_sql("SELECT * FROM person_dup;", engine)

print("Columnas origen:", len(df_origen.columns))
print("Filas encontradas en origen:", len(df_origen))

# =========================================================
# 3) PLANTILLA DESTINO (FLOWWW)
# =========================================================
df_plantilla = pd.read_csv(RUTA_PLANTILLA, sep=";", encoding="utf-8-sig")
columnas_destino = df_plantilla.columns.tolist()

print("Columnas destino:", len(columnas_destino))

# =========================================================
# 4) MAPEO ORIGEN → DESTINO
# =========================================================
mapeo = {
    "ClientID": "personaID",
    "ClientClinicID": "codclinica",
    "ClientName": "nombre",
    "ClientSurname1": "apellido1",
    "ClientSurname2": "apellido2",
    "ClientSex": "sexo",
    "ClientNIF": "dnicif",
    "ClientEmail": "email",
    "ClientBirthDate": "fechanacimiento",
    "ClientDate": "fechaalta",
    "ClientPhone1": "telefonofijo",
    "ClientPhone2": "movil",
    "ClientPhone3": "fax",
    "ClientAddress": "direccion",
    "ClientPostCosde": "codigopostal",
    "ClientCity": "poblacion",
    "ClientProvince": "provincia",
    #"ClientCustom1": "saldounidad",
}

# =========================================================
# 5) CONSTRUIR DATAFRAME FINAL
# =========================================================
df_final = pd.DataFrame(index=df_origen.index)

for col_dest in columnas_destino:
    if col_dest == "ClientGeneralConditions":
        df_final[col_dest] = "0"
        continue

    col_origen = mapeo.get(col_dest)

    if col_origen and col_origen in df_origen.columns:
        df_final[col_dest] = df_origen[col_origen]
    else:
        df_final[col_dest] = ""

# =========================================================
# 6) FORMATO FECHAS
# =========================================================
for col_fecha in ["ClientBirthDate", "ClientDate"]:
    if col_fecha in df_final.columns:
        df_final[col_fecha] = df_final[col_fecha].apply(convertir_fecha)

# =========================================================
# 7) LIMPIEZA TELÉFONOS
# =========================================================
for col in ["ClientPhone1", "ClientPhone2", "ClientPhone3"]:
    if col in df_final.columns:
        df_final[col] = df_final[col].apply(limpiar_telefono)

# =========================================================
# 8) NORMALIZAR SEXO
# =========================================================
if "ClientSex" in df_final.columns:
    df_final["ClientSex"] = df_final["ClientSex"].apply(normalizar_sexo)

# =========================================================
# 9) NORMALIZAR CP
# =========================================================
if "ClientPostCosde" in df_final.columns:
    df_final["ClientPostCosde"] = df_final["ClientPostCosde"].apply(normalizar_cp)

# =========================================================
# 10) LIMPIEZA NUMÉRICOS
# =========================================================
for col in ["ClientID", "ClinicID"]:
    if col in df_final.columns:
        df_final[col] = df_final[col].apply(limpiar_entero_como_texto)

# =========================================================
# 11) DEFAULT ClinicID (vacío/null/0 -> 1)
# =========================================================
if "ClientClinicID" in df_final.columns:
    df_final["ClientClinicID"] = df_final["ClientClinicID"].apply(lambda x: aplicar_default_clinicid(x, CLINICID_DEFAULT))

# Orden exacto según plantilla
df_final = df_final[columnas_destino]

# =========================================================
# 12) GUARDAR CSV FINAL
# =========================================================
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df_final.to_csv(
    RUTA_SALIDA,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ CSV generado correctamente:")
print(RUTA_SALIDA)
print(f"ℹ️ Regla aplicada: ClinicID vacío/null/0 => {CLINICID_DEFAULT}")
print("ℹ️ Mapeo aplicado: saldounidad -> ClientCustom1")
print("ℹ️ CP normalizado a 5 dígitos (ej: 08028)")

## Tabla diary_gen (agenda futura)

Requiere subir antes el mapeo de doctores.

Hay un registro sin profesional asignado: cuando el profesional es nulo, lo asigno por defecto al doctor 3119 (doctor titular).

In [ ]:
import os
import datetime
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# =========================================================
# CONFIG
# =========================================================
RUTA_SALIDA = r"C:\Migracion\ClinicaX\Destino\diarygen.csv"

# =========================================================
# UTILIDADES
# =========================================================
def convertir_fecha(v):
    if v is None or pd.isna(v):
        return ""
    try:
        return pd.to_datetime(v).strftime("%d/%m/%Y")
    except Exception:
        return ""

def convertir_hora(v):
    if v is None or pd.isna(v):
        return ""
    try:
        return pd.to_datetime(v).strftime("%H:%M")
    except Exception:
        return ""

# =========================================================
# ENV
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

# =========================================================
# 1) CONEXIÓN SQL SERVER (SQLAlchemy + Windows Auth)
# =========================================================
print("🔌 Conectando a SQL Server...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("✅ Conexión OK")

# =========================================================
# 2) QUERY SQL (YA CON MAPEOS HECHOS)
# =========================================================
query = """
SELECT
    a.codclinica,
    a.codgabinete,
    a.fecha,
    a.horacita,
    a.horafin,
    a.fechacreacion,
    a.anotaciones,

    cli.personaID  AS ClientID,
    prof.personaID AS DiaryGUserID

FROM agenda AS a
LEFT JOIN person_dup AS cli
    ON cli.codpersona = a.codpersona
   AND cli.codempresa = a.codempresa
LEFT JOIN person_dup AS prof
    ON prof.codpersona = a.codprofesional
   AND prof.codempresa = a.codempresa

WHERE a.codpersona <> '00000000-0000-0000-0000-000000000000'
    AND a.fecha >= DATEADD(DAY, 1, CAST(GETDATE() AS DATE));
"""

df = pd.read_sql(query, engine)

print(f"Filas recuperadas desde SQL: {len(df)}")

# =========================================================
# 3) CONSTRUIR DATAFRAME FINAL (FLOWW)
# =========================================================
df_final = pd.DataFrame()

df_final["DiaryGID"] = range(1, len(df) + 1)
df_final["DiaryID"]  = df_final["DiaryGID"]

df_final["DiaryGClientID"] = df["ClientID"]

df_final["DiaryGSUserID"] = 0
df_final["DiaryGUserID"] = df["DiaryGUserID"].fillna(3119).astype(int)

df_final["DiaryGClinicID"] = 1
df_final["DiaryGCabinetNum"] = 1

df_final["DiaryGDate"] = df["fecha"].apply(convertir_fecha)
df_final["DiaryGStart"] = df["horacita"].apply(convertir_hora)
df_final["DiaryGEnd"] = df["horafin"].apply(convertir_hora)

df_final["DiaryGCreated"] = df["fechacreacion"].apply(convertir_fecha)
df_final["DiaryGComments"] = df["anotaciones"]

df_final["DiaryZoneID"] = ""

# =========================================================
# 4) GUARDAR CSV
# =========================================================
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df_final.to_csv(
    RUTA_SALIDA,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("✅ CSV generado correctamente:")
print(RUTA_SALIDA)


## Tabla budgets_gen (presupuestos)

Requiere hacer antes el mapeo de tratamientos a servicios y productos.

`BudgetPrice` = `importebi` (importe sin IVA), de la vista SQL `Budget_gen`.

Al crear la tabla SQL uso el código de producto migrado 1466.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# =========================================================
# CONFIG
# =========================================================
RUTA_SALIDA = r"C:\Migracion\ClinicaX\Destino\budgets_gen.csv"

COLUMNAS_FINALES = [
    "BudgetGStatus",
    "BudgetGID",
    "BudgetID",
    "BudgetGClinicID",
    "BudgetGClientID",
    "BudgetGUserID",
    "BudgetGDoctorID",
    "BudgetGDate",
    "BudgetGValidity",
    "BudgetProductID",
    "BudgetUnits",
    "BudgetPrice",
    "BudgetDiscountAmount",
    "BudgetVAT",
    "BudgetComments",
]

# =========================================================
# ENV
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

# =========================================================
# 1) CONEXIÓN SQL SERVER (SQLAlchemy + Windows Auth)
# =========================================================
print("🔌 Conectando a SQL Server...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("✅ Conexión OK")

# =========================================================
# 2) LEER VISTA ORIGEN
# =========================================================
query = "SELECT * FROM budgets_gen;"
df = pd.read_sql(query, engine)

print(f"📊 Filas recuperadas: {len(df)}")
print(f"📊 Columnas origen: {list(df.columns)}")

# =========================================================
# 3) UTILIDADES
# =========================================================
def convertir_fecha(v):
    if v is None or pd.isna(v):
        return ""
    try:
        return pd.to_datetime(v).strftime("%d/%m/%Y")
    except Exception:
        return ""

def limpiar_entero_como_texto(x):
    if x is None or pd.isna(x):
        return ""
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    try:
        return str(int(float(s)))
    except Exception:
        return ""

# =========================================================
# 4) NORMALIZACIÓN / FORMATO
# =========================================================
for col in ["BudgetGID", "BudgetID", "BudgetClientID", "BudgetDoctorID", "BudgetUnits"]:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_entero_como_texto)

if "BudgetProductID" in df.columns:
    df["BudgetProductID"] = (
        pd.to_numeric(df["BudgetProductID"], errors="coerce")
        .fillna(2060)
        .astype(int)
    )

for col in ["BudgetPrice", "BudgetDiscount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).round(2)

if "BudgetCreateDate" in df.columns:
    df["BudgetGDate"] = df["BudgetCreateDate"].apply(convertir_fecha)
else:
    df["BudgetGDate"] = ""

# =========================================================
# 5) VALORES FIJOS FLOWWW
# =========================================================
df["BudgetGClinicID"] = 1
df["BudgetGUserID"] = 1
df["BudgetVAT"] = 0.21
df["BudgetGValidity"] = ""

# =========================================================
# 6) RENOMBRES A COLUMNAS FINALES
# =========================================================
df = df.rename(columns={
    "BudgetClientID": "BudgetGClientID",
    "BudgetDoctorID": "BudgetGDoctorID",
    "BudgetDiscount": "BudgetDiscountAmount",
})

# =========================================================
# 7) ASEGURAR COLUMNAS FINALES
# =========================================================
for col in COLUMNAS_FINALES:
    if col not in df.columns:
        df[col] = ""

# =========================================================
# 8) DEJAR SOLO LAS COLUMNAS PERMITIDAS
# =========================================================
df_final = df[COLUMNAS_FINALES].copy()

# =========================================================
# 9) GUARDAR CSV FINAL
# =========================================================
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df_final.to_csv(
    RUTA_SALIDA,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("✅ CSV 'budgets_gen' generado correctamente")
print("📌 Columnas finales:", list(df_final.columns))
print(RUTA_SALIDA)


## Tabla clients_crm (tabla SQL `historiaclinica`)

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# =========================================================
# CONFIG
# =========================================================
RUTA_SALIDA = r"C:\Migracion\ClinicaX\Destino\Clients_crm.csv"  # Ruta de salida del CSV

# =========================================================
# ENV
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

# =========================================================
# CONEXIÓN SQL SERVER (SQLAlchemy + Windows Auth)
# =========================================================
print("🔌 Conectando a SQL Server...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("✅ Conexión OK")

# =========================================================
# 2) LEER DATOS DE HISTORIA CLÍNICA Y CLIENTES
# =========================================================
query = """
   SELECT 
    p.personaID,
    p.nombre,
    p.apellido1,
    p.apellido2,
    h.*
FROM historiaclinica AS h
LEFT JOIN person_dup AS p
    ON h.codpersona = p.codpersona;
"""

df = pd.read_sql(query, engine)

print(f"📊 Filas recuperadas desde SQL: {len(df)}")
print(f"📊 Columnas recuperadas desde SQL: {list(df.columns)}")

# =========================================================
# 3) LIMPIEZA DE DATOS (Formato de fechas)
# =========================================================
def convertir_fecha(v):
    if v is None or pd.isna(v):
        return ""
    try:
        return pd.to_datetime(v).strftime("%d/%m/%Y")
    except Exception:
        return ""

for col_fecha in ["fechacreacion"]:
    if col_fecha in df.columns:
        df[col_fecha] = df[col_fecha].apply(convertir_fecha)

# =========================================================
# 4) MODIFICACIÓN DE DATOS
# =========================================================
df["ClientCID"] = range(1, len(df) + 1)
df["ClinicID"] = 1

df["ClientID"] = df["personaID"]
df["ClientCComments"] = df["anotaciones"]
df["ClientCDate"] = df["fechacreacion"]

df = df[['ClientCID', 'ClinicID', 'ClientID', 'ClientCComments', 'ClientCDate']]

# =========================================================
# 5) GUARDAR CSV FINAL
# =========================================================
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)

df.to_csv(
    RUTA_SALIDA,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("✅ CSV 'Clients_crm' generado correctamente:")
print(RUTA_SALIDA)

### clients_crm: generación y actualización del CSV (más la tabla `fichamedica`)

Este script genera y mantiene actualizado el fichero `Clients_crm.csv` para la migración de datos a Flowww del entorno de **Clínica X**.
El archivo funciona como **única fuente de verdad**: siempre se lee y se sobrescribe en la misma ubicación.

#### Qué hace
1. **Se conecta a SQL Server** con SQLAlchemy y **autenticación de Windows**.
2. **Lee el archivo `Clients_crm.csv` de la carpeta `Destino`**:
   - Si el archivo existe, lo carga como punto de partida.
   - Si no existe, crea un DataFrame vacío con la estructura que espera Flowww.
3. **Calcula el último `ClientCID`** existente para continuar la numeración sin duplicados.
4. **Ejecuta una consulta SQL** sobre las tablas `fichamedica` y `person_dup` para obtener estos campos:
   - `personaID`
   - `titulo`
   - `observaciones`
   - `fechacreacion`
5. **Limpia y normaliza los textos** (`titulo` y `observaciones`):
   - Decodifica las entidades HTML.
   - Elimina etiquetas HTML, saltos de línea, tabulaciones y caracteres invisibles.
   - Normaliza espacios y separadores.
6. **Construye el campo `ClientCComments`** concatenando:
   - `fichamedica.titulo`
   - `fichamedica.observaciones`
7. **Da a los datos el formato de Flowww**:
   - `ClientID` a partir de `personaID`.
   - `ClientCDate` a partir de `fechacreacion`, con formato `dd/mm/aaaa`.
   - `ClinicID` fijo (`1`).
   - `ClientCID` incremental a partir del último valor existente.
8. **Une los registros nuevos con los existentes** y **sobrescribe** el archivo `Destino/Clients_crm.csv`.

#### Resultado
El script deja actualizado `Clients_crm.csv`: conserva los registros anteriores, añade los nuevos con identificadores consecutivos y mantiene la trazabilidad del origen de los datos en SQL.

`fechamodificacion` se mapea a `ClientCDate`; si viene vacía, se rellena con `fechacreacion`.

In [ ]:
import os
import csv
import re
import html
import logging
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# =========================================================
# LOGGING (solo mensaje)
# =========================================================
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(message)s"))
logger.handlers = [handler]
logger.propagate = False

# =========================================================
# RUTAS (ÚNICA FUENTE DE VERDAD)
# =========================================================
DESTINO_DIR = r"C:\Migracion\ClinicaX\Destino"
OUTPUT_FILE = os.path.join(DESTINO_DIR, "Clients_crm.csv")

EXPECTED_COLS = [
    "ClientCID",
    "ClinicID",
    "ClientID",
    "ClientCComments",
    "ClientCDate"
]

# =========================================================
# ENV
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

# =========================================================
# CONEXIÓN SQL SERVER
# =========================================================
logger.info("🔌 Conectando a SQL Server con SQLAlchemy (Windows Auth)...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

logger.info("✅ Conexión OK")

# =========================================================
# 1) LEER Clients_crm.csv DESDE DESTINO
# =========================================================
if os.path.exists(OUTPUT_FILE):
    df_origen = pd.read_csv(
        OUTPUT_FILE,
        sep=";",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    df_origen.columns = [c.strip().replace("\ufeff", "") for c in df_origen.columns]
    logger.info("📂 Clients_crm.csv leído desde Destino")
else:
    logger.warning("⚠️ Clients_crm.csv no existe. Se creará uno nuevo.")
    df_origen = pd.DataFrame(columns=EXPECTED_COLS)

filas_antes = len(df_origen)

# =========================================================
# 2) ÚLTIMO ClientCID (SEGURO)
# =========================================================
if not df_origen.empty and "ClientCID" in df_origen.columns:
    last_clientcid = (
        pd.to_numeric(df_origen["ClientCID"], errors="coerce")
        .dropna()
        .astype(int)
        .max()
    )
    last_clientcid = int(last_clientcid) if pd.notna(last_clientcid) else 0
else:
    last_clientcid = 0

# =========================================================
# 3) QUERY SQL  (CAMBIO: fechamodificacion)
# =========================================================
query = """
    SELECT 
    p.personaID,
    f.titulo,
    f.observaciones,
    f.fechamodificacion,
    f.fechacreacion
FROM fichamedica AS f
 LEFT JOIN person_dup AS p
    ON f.codpersona = p.codpersona;
"""

df_sql = pd.read_sql(query, engine)

# =========================================================
# 4) LIMPIEZA FLOWWW (HTML / TEXTOS)
# =========================================================
def clean_text_for_flowww(value: str) -> str:
    if not value:
        return ""

    value = str(value)
    value = html.unescape(value)
    value = re.sub(r"<\s*br\s*/?>", " | ", value, flags=re.IGNORECASE)
    value = re.sub(r"<[^>]+>", "", value)
    value = re.sub(r"[\t\r\n]+", " | ", value)
    value = re.sub(r"[\u200b\u200c\u200d\uFEFF]", "", value)
    value = re.sub(r"\s{2,}", " ", value)
    value = re.sub(r"(\s\|\s)+", " | ", value)

    return value.strip(" |").strip()

def clean_date(v):
    d = pd.to_datetime(v, errors="coerce")
    return "" if pd.isna(d) else d.strftime("%d/%m/%Y")

# =========================================================
# 5) TRANSFORMACIÓN A FORMATO FLOWWW
# =========================================================
# =========================================================
# 5) TRANSFORMACIÓN A FORMATO FLOWWW
# =========================================================
df_sql["titulo"] = df_sql["titulo"].fillna("").apply(clean_text_for_flowww)
df_sql["observaciones"] = df_sql["observaciones"].fillna("").apply(clean_text_for_flowww)

df_sql["ClientCComments"] = (
    df_sql["titulo"].str.strip()
    + " - "
    + df_sql["observaciones"].str.strip()
)

# CAMBIO: ClientCDate = fechamodificacion si existe; si está vacío/null -> fechacreacion
# Normalizar vacíos en ambas fechas (por si vienen como '' o strings raras)
for col in ["fechamodificacion", "fechacreacion"]:
    if col in df_sql.columns:
        df_sql[col] = (
            df_sql[col]
            .astype(str)
            .str.strip()
            .replace({"": pd.NA, "None": pd.NA, "nan": pd.NA, "NaT": pd.NA})
        )

df_sql["ClientCDate"] = df_sql["fechamodificacion"].fillna(df_sql["fechacreacion"])

# Renombrar personaID -> ClientID (ya NO renombramos fechamodificacion)
df_sql = df_sql.rename(columns={
    "personaID": "ClientID"
})

df_sql["ClinicID"] = "1"
df_sql["ClientCID"] = range(last_clientcid + 1, last_clientcid + 1 + len(df_sql))
df_sql["ClientCDate"] = df_sql["ClientCDate"].apply(clean_date)

df_sql["ClientID"] = (
    pd.to_numeric(df_sql["ClientID"], errors="coerce")
    .fillna(0)
    .astype(int)
    .astype(str)
)

df_sql["ClientCID"] = df_sql["ClientCID"].astype(str)

df_sql = df_sql[EXPECTED_COLS]

# =========================================================
# 6) CONCATENAR + GUARDAR
# =========================================================
df_final = pd.concat([df_origen, df_sql], ignore_index=True)

os.makedirs(DESTINO_DIR, exist_ok=True)

df_final.to_csv(
    OUTPUT_FILE,
    index=False,
    sep=";",
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL
)

# =========================================================
# 7) OUTPUT FINAL
# =========================================================
logger.info(f"📄 Filas antes: {filas_antes}")
logger.info(f"➕ Registros añadidos: {len(df_sql)}")
logger.info(f"📊 Total final: {len(df_final)}")
logger.info("=" * 60)

## Tablas files y pictures, con auditoría

Genero `files.csv` y `pictures.csv` a partir de los metadatos extraídos de SQL. Descarto los ficheros sin extensión (o con una extensión solo numérica, como `.20`), conservo los acentos y limpio los nombres de fichero (por ejemplo, las comas sueltas entre guiones bajos).

La corrección evita usar `basename()` sobre `FileName`, porque algunos nombres contienen `/` dentro del propio texto, como `(S/H)`. Antes, eso hacía que el nombre se recortara mal y acabara como `H) – HYALDEW.pdf.pdf` (caso detectado en el cliente 675, con un nombre del tipo `<NOMBRE_PACIENTE>Firma_11_5_2022_HIALURONICO DENEB CLASSIC- (S/H) – HYALDEW.pdf.pdf`).

Ahora el script conserva el nombre original completo tal como viene de SQL, sin perder información, y así los scripts posteriores pueden cruzarlo correctamente con el fichero físico real.

In [ ]:
import os
import sys
import logging
import unicodedata
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
from pathlib import Path

MIGRACION_ROOT = Path(
    os.getenv("LIBS_ROOT", "../..")
)
if str(MIGRACION_ROOT) not in sys.path:
    sys.path.insert(0, str(MIGRACION_ROOT))

from Libs import *

# =========================================================
# LOGGING
# =========================================================
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(message)s"))
logger.handlers = [handler]
logger.propagate = False

# =========================================================
# RUTAS
# =========================================================
BASE_DIR = r"C:\Migracion\ClinicaX"
DESTINO_DIR = os.path.join(BASE_DIR, "Destino")

FILES_OUT = os.path.join(DESTINO_DIR, "files.csv")
PICTURES_OUT = os.path.join(DESTINO_DIR, "pictures.csv")

# ✅ AUDIT: ficheros sin extensión (o extensión numérica tipo .20)
AUDIT_OUT = os.path.join(DESTINO_DIR, "csv_audit_missing_extension.csv")

CSV_SEP = ";"
CSV_ENCODING = "cp1252"

DRY_RUN = False  # integrar con tu framework si ya existe

def file_basename(nombrearchivo: str) -> str:
    s = "" if nombrearchivo is None else str(nombrearchivo).strip()
    s = s.replace("\\", "/")
    return os.path.basename(s)[:255]

def keep_filename_raw(nombrearchivo: str) -> str:
    """
    Mantener el nombre exactamente como viene de SQL.
    NO usar basename porque algunos nombres contienen '/' como texto.
    """
    if nombrearchivo is None:
        return ""
    return str(nombrearchivo).strip()[:255]

# =========================================================
# CLASIFICACIÓN POR EXTENSIÓN (REGLA CLÍNICA)
# =========================================================
IMAGE_EXTS = {"jpg", "jpeg", "png", "bmp", "gif", "tif", "tiff", "heic"}

FILE_CLASS_MAP = {
    1: IMAGE_EXTS,
    2: {"doc", "docx", "rtf"},
    3: {"xls", "xlsx", "csv"},
    5: {"txt", "log"},
    6: {"zip", "rar", "7z"},
    7: {"pdf"},
}

def get_file_class_id(filename: str) -> int:
    ext = os.path.splitext(str(filename))[1].lower().replace(".", "")
    for cid, exts in FILE_CLASS_MAP.items():
        if ext in exts:
            return cid
    return 99

def is_missing_or_numeric_ext(filename: str) -> bool:
    """
    True si:
      - no tiene extensión
      - o la extensión es solo números (ej: .20)
    """
    s = "" if filename is None else str(filename).strip()
    if "." not in s:
        return True
    ext = s.rsplit(".", 1)[-1].strip()
    if ext == "":
        return True
    if ext.isdigit():
        return True
    return False

# =========================================================
# SQL
# =========================================================
load_dotenv(".env")

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

if not DB_SERVER or not DB_NAME:
    raise ValueError("Faltan variables DB_SERVER / DB_NAME")

logger.info("🔌 Conectando a SQL Server...")

odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={DB_SERVER};"
    f"Database={DB_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
    fast_executemany=True
)

# ✅ NUEVA QUERY (único cambio en origen)
QUERY = """
SELECT 
    p.personaID as FileClientID_PictureClientID,
    m.nombrearchivo as FileName,
    CONCAT('ficheros/', p.codnumpersona, '/', m.nombrearchivo) as FilePath_PicturePath,
    CAST(m.fechacreacion as date) as FileDate_PictureDate
FROM multimedia m
LEFT JOIN person_dup p 
    ON p.codpersona = m.codpersona
WHERE personaID is not null
"""

# =========================================================
# MAIN
# =========================================================
def flowww_safe_text(s):
    # Para comments u otros textos donde no importa perder tildes
    return clean_text_for_flowww_safe(
        s,
        ascii_only=True,
        strip_html=False
    )

def keep_sql_filename_exact(s):
    """
    Para FileName/FilePath/PicturePath:
    - preserva tildes y caracteres
    - normaliza a NFC para evitar \u0301 (acentos combinantes)
    - NO elimina ni modifica el contenido (solo normaliza unicode)
    """
    if s is None:
        return ""
    return unicodedata.normalize("NFC", str(s))

def fix_commas_in_filename(s: str) -> str:
    """
    ✅ 
    Limpiar SOLO la coma 'colada' en nombres tipo:
      - 'NOMBRE_,_APELLIDO.pdf' -> 'NOMBRE__APELLIDO.pdf'
      - 'APELLIDO,_NOMBRE.pdf' -> 'APELLIDO_NOMBRE.pdf'

    Reglas mínimas:
      - ',_'  -> '_'
      - '_,_' -> '__'
    """
    if s is None:
        return ""
    s = str(s)
    s = s.replace(",_", "_")
    s = s.replace("_,_", "__")
    return s

def main():
    os.makedirs(DESTINO_DIR, exist_ok=True)

    counters = {
        "sql_rows_total": 0,
        "rows_files_csv": 0,
        "rows_pictures_csv": 0,
        "no_match": 0,
        "total_files_csv": 0,
        "total_pictures_csv": 0,
        # ✅ contadores audit
        "rows_missing_ext_total": 0,
        "rows_missing_ext_unique_filenames": 0,
    }

    logger.info("📥 Ejecutando query SQL...")
    df = pd.read_sql(text(QUERY), engine)
    counters["sql_rows_total"] = len(df)

    if df.empty:
        logger.info("⚠️ Query sin resultados")
        logger.info("\n=== CONTADORES ===")
        for k in sorted(counters.keys()):
            logger.info(f"{k}: {counters[k]}")
        return

    # Columnas que vienen del SQL
    df.columns = [
        "FileClientID_PictureClientID",
        "FileName",
        "FilePath_PicturePath",
        "FileDate_PictureDate",
    ]

    # Mantener el nombre original EXACTO; no usar basename
    df["FileName"] = df["FileName"].map(keep_filename_raw)

    # Path ya viene construido en SQL; normalizamos separador
    df["FlowwwPath"] = (
        df["FilePath_PicturePath"]
        .astype(str)
        .str.replace("\\", "/", regex=False)
        .str.strip()
    )

    # Extensión para separar pics/files y para FileClassID
    df["ext"] = (
        df["FileName"]
        .astype(str)
        .str.rsplit(".", n=1).str[-1]
        .str.lower()
        .where(df["FileName"].astype(str).str.contains(r"\.", regex=True), "")
    )

    # Mantener normalización de fecha con función existente + formato DD-MM-YYYY
    df["FechaNorm"] = df["FileDate_PictureDate"].map(normalizar_timestamp_yyyymmddhhmmss)
    df["FechaNorm"] = pd.to_datetime(df["FechaNorm"], errors="coerce").dt.strftime("%d-%m-%Y")

    # =========================================================
    # ✅ AUDIT: ficheros sin extensión (o ext numérica .20)
    # =========================================================
    mask_missing_ext = df["FileName"].map(is_missing_or_numeric_ext)

    df_audit = df.loc[mask_missing_ext, [
        "FileClientID_PictureClientID",
        "FileName",
        "FlowwwPath",
        "FechaNorm",
    ]].copy()

    counters["rows_missing_ext_total"] = int(mask_missing_ext.sum())
    counters["rows_missing_ext_unique_filenames"] = int(df_audit["FileName"].nunique(dropna=True))

    if not df_audit.empty and not DRY_RUN:
        df_audit.rename(columns={
            "FileClientID_PictureClientID": "ClientID",
        }, inplace=True)

        df_audit.to_csv(
            AUDIT_OUT,
            sep=CSV_SEP,
            encoding=CSV_ENCODING,
            index=False,
            na_rep="\\N",
            lineterminator="\n",
        )

    # ✅ IMPORTANTÍSIMO: lo que va a AUDIT no entra en outputs
    df = df.loc[~mask_missing_ext].copy()

    # =========================================================
    # Separación igual que antes (sobre df ya filtrado)
    # =========================================================
    df_pictures = df[df["ext"].isin(IMAGE_EXTS)].copy()
    df_files = df[~df["ext"].isin(IMAGE_EXTS)].copy()

    counters["rows_files_csv"] = len(df_files)
    counters["rows_pictures_csv"] = len(df_pictures)

    files_df = pd.DataFrame({
        "FileID": range(1, len(df_files) + 1),
        "FileClientID": df_files["FileClientID_PictureClientID"].astype(str),
        # ✅ NO ASCII: preservar exactamente (con NFC) + ✅ limpiar coma colada
        "FileName": df_files["FileName"]
            .map(keep_sql_filename_exact)
            .map(fix_commas_in_filename)
            .str.slice(0, 255),
        "FileClassID": df_files["FileName"].map(get_file_class_id).astype(str),
        # ✅ NO ASCII: preservar exactamente (con NFC) + ✅ limpiar coma colada
        "FilePath": df_files["FlowwwPath"]
            .map(keep_sql_filename_exact)
            .map(fix_commas_in_filename)
            .str.slice(0, 255),
        "FileDate": df_files["FechaNorm"],
        "FileComments": "",
    })

    counters["total_files_csv"] = len(files_df)

    pictures_df = pd.DataFrame({
        "PictureID": range(1, len(df_pictures) + 1),
        "PictureClientID": df_pictures["FileClientID_PictureClientID"].astype(str),
        # ✅ ÚNICO CAMBIO: poner el nombre del fichero (foto) en PictureComments
        "PictureComments": df_pictures["FileName"].map(keep_sql_filename_exact).str.slice(0, 255),
        "PictureDate": df_pictures["FechaNorm"],
        # ✅ NO ASCII: preservar exactamente (con NFC) + ✅ limpiar coma colada (por si aplica)
        "PicturePath": df_pictures["FlowwwPath"]
            .map(keep_sql_filename_exact)
            .map(fix_commas_in_filename)
            .str.slice(0, 255),
    })

    counters["total_pictures_csv"] = len(pictures_df)

    if not DRY_RUN:
        files_df.to_csv(
            FILES_OUT,
            sep=CSV_SEP,
            encoding=CSV_ENCODING,
            index=False,
            na_rep="\\N",
            lineterminator="\n",
        )
        pictures_df.to_csv(
            PICTURES_OUT,
            sep=CSV_SEP,
            encoding=CSV_ENCODING,
            index=False,
            na_rep="\\N",
            lineterminator="\n",
        )

    logger.info("\n=== CONTADORES ===")
    for k in sorted(counters.keys()):
        logger.info(f"{k}: {counters[k]}")

    logger.info(f"\nAUDIT missing extension: {AUDIT_OUT}")
    logger.info("\n✅ Proceso finalizado" + (" (DRY_RUN)" if DRY_RUN else ""))

if __name__ == "__main__":
    main()

## Tabla deferred_uninvoiced (pagos aplazados)

In [ ]:
import os
import sys
import re
import csv
import warnings
import logging
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from urllib.parse import quote_plus
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.exc import SAWarning
from dotenv import load_dotenv

# =========================================================
# IMPORT UTILS (NO se modifica)
# =========================================================
MIGRACION_ROOT = Path(
    os.getenv("LIBS_ROOT", "../..")
)
if str(MIGRACION_ROOT) not in sys.path:
    sys.path.insert(0, str(MIGRACION_ROOT))

try:
    from Libs import *  # noqa
except Exception:
    pass

# =========================================================
# SILENCIAR WARNING SQLALCHEMY (opcional)
# =========================================================
warnings.filterwarnings(
    "ignore",
    category=SAWarning,
    message=r"Unrecognized server version info.*"
)

# =========================================================
# LOGGING
# =========================================================
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("deffered_uninvoices")

# =========================================================
# CONFIG
# =========================================================
OUTPUT_DIR = Path(r"C:\Migracion\ClinicaX\Destino")
OUTPUT_FILE = OUTPUT_DIR / "deffered_uninvoices.csv"
SQL_QUERY = "SELECT * FROM deudores"

OUTPUT_COLS = [
    "DeferredUID",
    "DeferredUTicketGID",
    "DeferredUTicketID",
    "DeferredUNumber",
    "DeferredUYear",
    "DeferredUAmount",
    "DeferredUSettlement",
    "DeferredUPTicketGID",
    "DeferredUDeleted",
    "DeferredUPClinicID",
    "DeferredUPDate",
    "DeferredUPTime",
    "DeferredUImportID",
    "DeferredUMigratedClinicID",
    "DeferredUMigratedClientID",
    "DeferredUMigratedVAT",
    "DeferredUMigratedDate",
    "DeferredUMigratedDesc",
]

# =========================================================
# HELPERS
# =========================================================
def norm_key(s: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "", str(s)).lower()

def find_col(df: pd.DataFrame, wanted: str) -> str:
    wanted_k = norm_key(wanted)
    col_map = {norm_key(c): c for c in df.columns}
    if wanted_k not in col_map:
        raise KeyError(
            f"No encuentro la columna '{wanted}' en origen. Disponibles: {list(df.columns)}"
        )
    return col_map[wanted_k]

def to_str_series(s: pd.Series) -> pd.Series:
    return s.astype(str).replace({"None": "", "nan": ""}).fillna("").str.strip()

def parse_datetime_series(series: pd.Series) -> pd.Series:
    """Parse robusto SIN warnings: intenta formatos típicos en cascada."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return series

    s = to_str_series(series)

    dt = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S.%f", errors="coerce")
    m = dt.isna() & s.ne("")
    if m.any():
        dt2 = pd.to_datetime(s[m], format="%Y-%m-%d %H:%M:%S", errors="coerce")
        dt.loc[m] = dt2

    m = dt.isna() & s.ne("")
    if m.any():
        dt3 = pd.to_datetime(s[m], format="%Y-%m-%d", errors="coerce")
        dt.loc[m] = dt3

    m = dt.isna() & s.ne("")
    if m.any():
        dt4 = pd.to_datetime(s[m], format="%d/%m/%Y", errors="coerce")
        dt.loc[m] = dt4

    return dt

def format_time_hhmmss(series: pd.Series) -> pd.Series:
    dt = parse_datetime_series(series)
    out = dt.dt.strftime("%H:%M:%S").fillna("")

    # fallback si venía ya como hora en texto
    s = to_str_series(series)
    s2 = s.str.replace(r"^(\d{1,2}):(\d{2})$", r"\1:\2:00", regex=True)
    s2 = s2.str.replace(r"^(\d):", r"0\1:", regex=True)
    ok = s2.str.match(r"^\d{2}:\d{2}:\d{2}$")

    return out.where(~(out.eq("") & ok), s2).fillna("")

def _parse_decimal_eu(x: str) -> Decimal:
    x = x.strip()
    if x == "":
        raise InvalidOperation
    x = x.replace("€", "").replace(" ", "")
    x = x.replace(".", "").replace(",", ".")
    return Decimal(x)

def format_amount_eur(series: pd.Series) -> pd.Series:
    s = to_str_series(series)

    def fmt_one(x: str) -> str:
        if x == "":
            return ""
        try:
            d = _parse_decimal_eu(x)
            d = abs(d)  # quitar negativos
            d = d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        except (InvalidOperation, ValueError):
            return x
        return f"{d:.2f}".replace(".", ",") + " €"

    return s.apply(fmt_one)

# =========================================================
# SQL CONNECTION
# =========================================================
def build_engine():
    load_dotenv(".env")

    DB_SERVER = os.getenv("DB_SERVER")
    DB_NAME = os.getenv("DB_NAME")

    if not DB_SERVER or not DB_NAME:
        raise ValueError("Faltan variables DB_SERVER / DB_NAME")

    logger.info("🔌 Conectando a SQL Server...")

    odbc_str = (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server={DB_SERVER};"
        f"Database={DB_NAME};"
        "Trusted_Connection=yes;"
        "TrustServerCertificate=yes;"
    )

    return create_engine(
        "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
        fast_executemany=True
    )

# =========================================================
# MAIN
# =========================================================
def main():
    engine = build_engine()

    logger.info("📥 Leyendo origen: deudores")
    df_src = pd.read_sql(SQL_QUERY, engine)
    origen_count = len(df_src)
    logger.info("✅ Registros origen (SQL): %d", origen_count)
    logger.info("🧾 Columnas origen: %s", list(df_src.columns))

    # localizar columnas origen (robusto a espacios/mayúsculas)
    col_persona_id = find_col(df_src, "personaID")
    col_fecha_deuda = find_col(df_src, "fechadeuda")
    col_fecha_deuda_dt = find_col(df_src, "fechadeudadatetime")
    col_saldo = find_col(df_src, "saldoclinica")

    # ordenar para autoincrement estable
    df_src = df_src.copy()
    df_src["_persona_sort"] = to_str_series(df_src[col_persona_id])
    df_src["_dt_sort"] = parse_datetime_series(df_src[col_fecha_deuda_dt])
    df_src = df_src.sort_values(["_persona_sort", "_dt_sort"], kind="mergesort").reset_index(drop=True)

    n = len(df_src)
    df_out = pd.DataFrame({c: [""] * n for c in OUTPUT_COLS})

    # Autoincrement desde 1
    seq = pd.Series(range(1, n + 1), dtype="int64").astype(str)
    df_out["DeferredUID"] = seq
    df_out["DeferredUNumber"] = seq
    df_out["DeferredUImportID"] = seq  # autoincrement también

    # Fijos
    df_out["DeferredUTicketGID"] = "0"
    df_out["DeferredUTicketID"] = "0"
    df_out["DeferredUPTicketGID"] = "0"
    df_out["DeferredUDeleted"] = "0"
    df_out["DeferredUPClinicID"] = "1"
    df_out["DeferredUMigratedClinicID"] = "1"
    df_out["DeferredUMigratedVAT"] = "0,21"
    df_out["DeferredUMigratedDesc"] = ""

    # Mapeos
    df_out["DeferredUMigratedClientID"] = to_str_series(df_src[col_persona_id])

    # Fechas desde fechadeuda
    dt_deuda = parse_datetime_series(df_src[col_fecha_deuda])
    fecha_ddmmyyyy = dt_deuda.dt.strftime("%d/%m/%Y").fillna("")
    # si algo no parsea, conserva texto (no inventar)
    s_fecha = to_str_series(df_src[col_fecha_deuda])
    fecha_ddmmyyyy = fecha_ddmmyyyy.where(~(dt_deuda.isna() & s_fecha.ne("")), s_fecha)

    df_out["DeferredUPDate"] = fecha_ddmmyyyy
    df_out["DeferredUSettlement"] = fecha_ddmmyyyy
    df_out["DeferredUMigratedDate"] = fecha_ddmmyyyy

    # Año desde fechadeuda (FIX: sin fillna("") antes de Int64)
    year = dt_deuda.dt.year.astype("Int64")
    df_out["DeferredUYear"] = year.astype(str).replace("<NA>", "")

    # Hora desde fechadeudadatetime (solo hora)
    dt_deuda_dt = parse_datetime_series(df_src[col_fecha_deuda_dt])
    df_out["DeferredUPTime"] = dt_deuda_dt.dt.strftime("%H:%M:%S").fillna("")
    # fallback si viniera como hora en texto
    if df_out["DeferredUPTime"].eq("").any():
        df_out["DeferredUPTime"] = df_out["DeferredUPTime"].where(
            df_out["DeferredUPTime"].ne(""),
            format_time_hhmmss(df_src[col_fecha_deuda_dt])
        )

    # Importe desde saldoclinica (quitando negativos)
    df_out["DeferredUAmount"] = format_amount_eur(df_src[col_saldo])

    # Contadores
    csv_count = len(df_out)
    logger.info("📊 Registros a escribir (CSV): %d", csv_count)
    logger.info("✅ Resumen: origen(SQL)=%d | salida(CSV)=%d", origen_count, csv_count)

    # Export
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(
        OUTPUT_FILE,
        index=False,
        sep=";",
        encoding="utf-8-sig",
        lineterminator="\r\n",
        quoting=csv.QUOTE_MINIMAL,
    )

    logger.info("📄 CSV generado: %s", str(OUTPUT_FILE))

if __name__ == "__main__":
    main()

## Tabla advances_gen (acreedores)

In [ ]:
import os
import sys
import re
import csv
import warnings
import logging
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from urllib.parse import quote_plus
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.exc import SAWarning
from dotenv import load_dotenv

# =========================================================
# IMPORT UTILS (NO se modifica)
# =========================================================
MIGRACION_ROOT = Path(
    os.getenv("LIBS_ROOT", "../..")
)
if str(MIGRACION_ROOT) not in sys.path:
    sys.path.insert(0, str(MIGRACION_ROOT))

try:
    from Libs import *  # noqa
except Exception:
    pass

# =========================================================
# SILENCIAR WARNING SQLALCHEMY (opcional)
# =========================================================
warnings.filterwarnings(
    "ignore",
    category=SAWarning,
    message=r"Unrecognized server version info.*"
)

# =========================================================
# LOGGING
# =========================================================
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("advances_gen")

# =========================================================
# CONFIG
# =========================================================
OUTPUT_DIR = Path(r"C:\Migracion\ClinicaX\Destino")
OUTPUT_FILE = OUTPUT_DIR / "advances_gen.csv"
SQL_QUERY = "SELECT * FROM acreedores"  # Cambié la consulta SQL a acreedores

OUTPUT_COLS = [
    "AdvanceGID",
    "AdvanceGDate",
    "AdvanceGClientID",
    "AdvanceGClinicID",
    "AdvanceGBudgetGID",
    "AdvanceAmount",
    "AdvanceVAT",
]

# =========================================================
# HELPERS
# =========================================================
def norm_key(s: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "", str(s)).lower()

def find_col(df: pd.DataFrame, wanted: str) -> str:
    wanted_k = norm_key(wanted)
    col_map = {norm_key(c): c for c in df.columns}
    if wanted_k not in col_map:
        raise KeyError(
            f"No encuentro la columna '{wanted}' en origen. Disponibles: {list(df.columns)}"
        )
    return col_map[wanted_k]

def to_str_series(s: pd.Series) -> pd.Series:
    return s.astype(str).replace({"None": "", "nan": ""}).fillna("").str.strip()

def parse_datetime_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        return series

    s = to_str_series(series)

    dt = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S.%f", errors="coerce")
    m = dt.isna() & s.ne("")
    if m.any():
        dt2 = pd.to_datetime(s[m], format="%Y-%m-%d %H:%M:%S", errors="coerce")
        dt.loc[m] = dt2

    m = dt.isna() & s.ne("")
    if m.any():
        dt3 = pd.to_datetime(s[m], format="%Y-%m-%d", errors="coerce")
        dt.loc[m] = dt3

    m = dt.isna() & s.ne("")
    if m.any():
        dt4 = pd.to_datetime(s[m], format="%d/%m/%Y", errors="coerce")
        dt.loc[m] = dt4

    return dt

def format_time_hhmmss(series: pd.Series) -> pd.Series:
    dt = parse_datetime_series(series)
    out = dt.dt.strftime("%H:%M:%S").fillna("")

    s = to_str_series(series)
    s2 = s.str.replace(r"^(\d{1,2}):(\d{2})$", r"\1:\2:00", regex=True)
    s2 = s2.str.replace(r"^(\d):", r"0\1:", regex=True)
    ok = s2.str.match(r"^\d{2}:\d{2}:\d{2}$")

    return out.where(~(out.eq("") & ok), s2).fillna("")

def _parse_decimal_eu(x: str) -> Decimal:
    x = x.strip()
    if x == "":
        raise InvalidOperation
    x = x.replace("€", "").replace(" ", "")
    x = x.replace(".", "").replace(",", ".")
    return Decimal(x)

def format_amount_eur(series: pd.Series) -> pd.Series:
    s = to_str_series(series)

    def fmt_one(x: str) -> str:
        if x == "":
            return ""
        try:
            d = _parse_decimal_eu(x)
            d = abs(d)  # quitar negativos
            d = d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        except (InvalidOperation, ValueError):
            return x
        return f"{d:.2f}".replace(".", ",")

    return s.apply(fmt_one)

# =========================================================
# SQL CONNECTION
# =========================================================
def build_engine():
    load_dotenv(".env")

    DB_SERVER = os.getenv("DB_SERVER")
    DB_NAME = os.getenv("DB_NAME")

    if not DB_SERVER or not DB_NAME:
        raise ValueError("Faltan variables DB_SERVER / DB_NAME")

    logger.info("🔌 Conectando a SQL Server...")

    odbc_str = (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server={DB_SERVER};"
        f"Database={DB_NAME};"
        "Trusted_Connection=yes;"
        "TrustServerCertificate=yes;"
    )

    return create_engine(
        "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
        fast_executemany=True
    )

# =========================================================
# MAIN
# =========================================================
def main():
    engine = build_engine()

    logger.info("📥 Leyendo origen: acreedores")
    df_src = pd.read_sql(SQL_QUERY, engine)
    origen_count = len(df_src)
    logger.info("✅ Registros origen (SQL): %d", origen_count)
    logger.info("🧾 Columnas origen: %s", list(df_src.columns))

    # localizar columnas origen (robusto a espacios/mayúsculas)
    col_persona_id = find_col(df_src, "PersonaID")
    col_fecha_credito = find_col(df_src, "fechacredito")
    col_saldo = find_col(df_src, "saldoclinica")

    # ordenar para autoincrement estable
    df_src = df_src.copy()
    df_src["_persona_sort"] = to_str_series(df_src[col_persona_id])
    df_src["_dt_sort"] = parse_datetime_series(df_src[col_fecha_credito])
    df_src = df_src.sort_values(["_persona_sort", "_dt_sort"], kind="mergesort").reset_index(drop=True)

    n = len(df_src)
    df_out = pd.DataFrame({c: [""] * n for c in OUTPUT_COLS})

    # Autoincrement desde 1
    seq = pd.Series(range(1, n + 1), dtype="int64").astype(str)
    df_out["AdvanceGID"] = seq

    # Fijos
    df_out["AdvanceGClinicID"] = "1"
    df_out["AdvanceGBudgetGID"] = "0"
    df_out["AdvanceVAT"] = "0.21"

    # Mapeos
    df_out["AdvanceGClientID"] = to_str_series(df_src[col_persona_id])

    # Fechas desde fechacredito
    dt_credito = parse_datetime_series(df_src[col_fecha_credito])
    fecha_ddmmyyyy = dt_credito.dt.strftime("%d/%m/%Y").fillna("")
    s_fecha = to_str_series(df_src[col_fecha_credito])
    fecha_ddmmyyyy = fecha_ddmmyyyy.where(~(dt_credito.isna() & s_fecha.ne("")), s_fecha)

    df_out["AdvanceGDate"] = fecha_ddmmyyyy

    # Importe desde saldoclinica
    df_out["AdvanceAmount"] = format_amount_eur(df_src[col_saldo])

    # Contadores
    csv_count = len(df_out)
    logger.info("📊 Registros a escribir (CSV): %d", csv_count)
    logger.info("✅ Resumen: origen(SQL)=%d | salida(CSV)=%d", origen_count, csv_count)

    # Export
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(
        OUTPUT_FILE,
        index=False,
        sep=";",
        encoding="utf-8-sig",
        lineterminator="\r\n",
        quoting=csv.QUOTE_MINIMAL,
    )

    logger.info("📄 CSV generado: %s", str(OUTPUT_FILE))

if __name__ == "__main__":
    main()

## Tablas clients_tracking_gen (parámetros corporales, estudio antropométrico) y diary_gen_hidden (citas ficticias invisibles)

El `DiaryID` se fija a partir del último `DiaryID` de `diary_gen` ya cargado.

Este script extrae datos de SQL Server (`parametros_corporales`) y genera dos CSV para Flowww: `clients_tracking_gen.csv` y `diary_gen_hidden.csv`.

En `clients_tracking_gen.csv` crea registros de seguimiento con `TrackGClinicID = 1`: añade una fila con la **fecha de lectura** (`TrackParamID = 1`), genera los parámetros fijos (`peso`, `talla`, `imc`, etc.), después una fila descriptiva de la **parte del cuerpo** (`TrackParamID = 10`, con `TrackValue = parte_del_cuerpo`) y, por último, una fila por cada medida (`medida1`, `medida2`, `medida3`, `medidax`) con su `ParamID` correspondiente.

A partir de ese seguimiento construye `diary_gen_hidden.csv`, con **una cita por cada `TrackGID` distinto**. El `DiaryID` coincide con `TrackGDiaryID` para mantener la referencia entre ambos CSV, y la numeración empieza en `300` porque ya existían registros en `diary_gen`.
**`DiaryGID` también coincide con `DiaryID` y con `TrackGDiaryID`**, de modo que los tres identificadores quedan alineados en el mismo rango.

Si varios `TrackGID` comparten la misma `TrackGDate`, las citas se crean en esa fecha, desplazando cada una **30 minutos** respecto a la anterior.

Al final exporta ambos CSV en la ruta de destino y muestra por consola los contadores y las trazas.

In [ ]:
# =========================================================
# CLIENTS TRACKING GENERATOR
# SQL Server -> CSV Flowww
# Script reusable para migraciones tabulares
# =========================================================

# =========================================================
# IMPORTS
# =========================================================
import os
import sys
import logging
import pandas as pd
import numpy as np

from pathlib import Path
from urllib.parse import quote_plus
from datetime import datetime, timedelta

from sqlalchemy import create_engine
from sqlalchemy.exc import SAWarning
from dotenv import load_dotenv

import warnings
warnings.filterwarnings("ignore", category=SAWarning)

# =========================================================
# IMPORT UTILS
# =========================================================
MIGRACION_ROOT = Path(
    os.getenv("LIBS_ROOT", "../..")
)
if str(MIGRACION_ROOT) not in sys.path:
    sys.path.insert(0, str(MIGRACION_ROOT))

try:
    from Libs import *  # noqa
except Exception:
    pass


# =========================================================
# CONFIGURACIÓN
# =========================================================

# -------- SQL SOURCE --------
SQL_QUERY = """
SELECT *
FROM parametros_corporales
"""

SOURCE_TABLE_NAME = "parametros_corporales"

# -------- COLUMNAS ORIGEN NECESARIAS --------
ORIG_COLS = [
    "ClientID",
    "fecha",
    "peso",
    "talla",
    "imc",
    "pesodeseable",
    "get_kcal_dia",
    "actividad_fisica",
    "complexion_desc",
    "estres",
    "parte_del_cuerpo",
    "medida1",
    "medida2",
    "medida3",
    "medidax",
]

# -------- DESTINO --------
OUTPUT_DIR = Path(r"C:\Migracion\ClinicaX\Destino")

TRACKING_OUTPUT_FILENAME = "clients_tracking_gen.csv"
DIARY_OUTPUT_FILENAME = "diary_gen_hidden.csv"

CSV_SEPARATOR = ";"
CSV_ENCODING = "utf-8-sig"

# =========================================================
# ESQUEMA DESTINO TRACKING
# =========================================================

DEST_COLS = [
    "TrackGID",
    "TrackGClinicID",
    "TrackGClientID",
    "TrackGDiaryID",
    "TrackGDate",
    "TrackGTime",
    "TrackID",
    "TrackParamID",
    "TrackValue",
]

TRACK_GCLINIC_ID = 1

# =========================================================
# ESQUEMA DESTINO DIARY
# =========================================================

DIARY_DEST_COLS = [
    "DiaryGID",
    "DiaryID",
    "DiaryGClientID",
    "DiaryGClinicID",
    "DiaryGDate",
    "DiaryGStart",
    "DiaryGEnd",
]

DIARY_GCLINIC_ID = 1

# =========================================================
# PARÁMETROS FIJOS
# =========================================================

FIXED_PARAMS = {
    "peso": 2,
    "talla": 3,
    "imc": 4,
    "pesodeseable": 5,
    "get_kcal_dia": 6,
    "actividad_fisica": 7,
    "complexion_desc": 8,
    "estres": 9,
}

# =========================================================
# PARAM IDs ESPECIALES
# =========================================================

DATE_PARAM_ID = 1
BODY_PART_PARAM_ID = 10

# =========================================================
# PARAM ID PARTE DEL CUERPO (VALOR LITERAL)
# =========================================================
BODY_PART_PARAM_MAP = {
    "Abdominal": 15,
    "Antebrazo (máximo)": 16,
    "Brazo (flexionado)": 17,
    "Brazo (relajado)": 18,
    "Bíceps": 19,
    "Cabeza": 20,
    "Cadera": 21,
    "Cintura": 22,
    "Cresto ilíaca": 23,
    "Cuello": 24,
    "Muslo (supra-rotulo)": 25,
    "Muñeca (estiloideo distal)": 26,
    "Pantorrilla (máxima)": 27,
    "Pantorrilla medial": 28,
    "Subescapular": 29,
    "Supraespinal": 30,
    "Tobillo (mínimo)": 31,
    "Tríceps": 32,
    "Tórax (mesoesternal)": 33,
    "Tórax transverso": 34,
}

BODY_PART_ALIASES = {
    "Muñeca(estiloideo distal)": "Muñeca (estiloideo distal)",
    "Tórax transverso.": "Tórax transverso",
}

# =========================================================
# PARÁMETROS MEDIDAS CORPORALES
# =========================================================
MEASURE_PARAMS = {
    "medida1": 11,
    "medida2": 12,
    "medida3": 13,
    "medidax": 14,
}

# =========================================================
# LOGGING
# =========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)

# =========================================================
# HELPERS
# =========================================================

def clean_invalid_characters(value):
    if value is None:
        return ""
    s = str(value)
    s = s.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    return s.strip()


def clean_text(value, max_len=255):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""

    s = str(value).strip()

    if s.lower() in ("nan", "none", "null"):
        return ""

    s = clean_invalid_characters(s)

    if len(s) > max_len:
        s = s[:max_len]

    return s


def safe_to_datetime(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, errors="coerce", dayfirst=False)

    mask = dt.isna()
    if mask.any():
        dt2 = pd.to_datetime(series[mask], errors="coerce", dayfirst=True)
        dt.loc[mask] = dt2

    mask = dt.isna()
    if mask.any():
        truncated = series[mask].astype(str).str[:10]
        dt3 = pd.to_datetime(truncated, errors="coerce", dayfirst=True)
        dt.loc[mask] = dt3

    return dt


def format_date_ddmmyyyy(value) -> str:
    if pd.isna(value):
        return ""

    try:
        return value.strftime("%d/%m/%Y")
    except Exception:
        return clean_text(value, 10)


def extract_time_hhmm(value) -> str:
    if pd.isna(value):
        return ""

    try:
        return value.strftime("%H:%M")
    except Exception:
        return ""


def add_30_minutes_hhmm(time_value: str) -> str:
    time_value = clean_text(time_value, 5)
    if time_value == "":
        return ""

    try:
        dt = datetime.strptime(time_value, "%H:%M")
        return (dt + timedelta(minutes=30)).strftime("%H:%M")
    except Exception:
        return ""


def add_minutes_hhmm(time_value: str, minutes: int) -> str:
    time_value = clean_text(time_value, 5)
    if time_value == "":
        return ""

    try:
        dt = datetime.strptime(time_value, "%H:%M")
        return (dt + timedelta(minutes=minutes)).strftime("%H:%M")
    except Exception:
        return ""


def normalize_body_part_name(value) -> str:
    cleaned = clean_text(value, 255)
    if cleaned == "":
        return ""

    return BODY_PART_ALIASES.get(cleaned, cleaned)


# =========================================================
# SQL CONNECTION
# =========================================================

def build_engine():
    load_dotenv(".env")

    DB_SERVER = os.getenv("DB_SERVER")
    DB_NAME = os.getenv("DB_NAME")

    if not DB_SERVER or not DB_NAME:
        raise ValueError("Faltan variables DB_SERVER / DB_NAME")

    logger.info("🔌 Conectando a SQL Server...")

    odbc_str = (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server={DB_SERVER};"
        f"Database={DB_NAME};"
        "Trusted_Connection=yes;"
        "TrustServerCertificate=yes;"
    )

    return create_engine(
        "mssql+pyodbc:///?odbc_connect=" + quote_plus(odbc_str),
        fast_executemany=True
    )


# =========================================================
# VALIDACIÓN COLUMNAS ORIGEN
# =========================================================

def validate_source_columns(df: pd.DataFrame):
    detected = set(df.columns)
    required = set(ORIG_COLS)

    missing = required - detected

    if missing:
        raise KeyError(
            f"Columnas faltantes en origen: {sorted(missing)}\n"
            f"Columnas detectadas: {sorted(detected)}"
        )


def validate_body_part_mapping(df_src: pd.DataFrame):
    body_parts = (
        df_src["parte_del_cuerpo"]
        .astype(str)
        .map(normalize_body_part_name)
        .replace({"": np.nan})
        .dropna()
        .unique()
        .tolist()
    )

    unmapped = [value for value in body_parts if value not in BODY_PART_PARAM_MAP]

    if unmapped:
        raise KeyError(
            "Faltan partes del cuerpo en BODY_PART_PARAM_MAP: "
            f"{sorted(unmapped)}"
        )


# =========================================================
# CONSTRUCCIÓN DESTINO TRACKING
# =========================================================

def build_dest_df(df_src: pd.DataFrame) -> pd.DataFrame:

    records = []

    track_gid = 1
    track_diary = 300  # ---> cambiar el valor en base a ultimo diaryID de diary_gen ( ultimo id de cita creada)
    track_id = 1

    date_counter = 0
    fixed_counter = 0
    body_part_counter = 0
    measure_counter = 0

    for _, row in df_src.iterrows():

        client_id = row["ClientID"]
        fecha_dt = row["_fecha_dt"]

        track_date = format_date_ddmmyyyy(fecha_dt)
        track_time = extract_time_hhmm(fecha_dt)

        if track_date != "":
            records.append({
                "TrackGID": track_gid,
                "TrackGClinicID": TRACK_GCLINIC_ID,
                "TrackGClientID": client_id,
                "TrackGDiaryID": track_diary,
                "TrackGDate": track_date,
                "TrackGTime": track_time,
                "TrackID": track_id,
                "TrackParamID": DATE_PARAM_ID,
                "TrackValue": track_date,
            })
            track_id += 1
            date_counter += 1

        for col, param_id in FIXED_PARAMS.items():
            value = row[col]

            if pd.isna(value) or str(value).strip() == "":
                continue

            records.append({
                "TrackGID": track_gid,
                "TrackGClinicID": TRACK_GCLINIC_ID,
                "TrackGClientID": client_id,
                "TrackGDiaryID": track_diary,
                "TrackGDate": track_date,
                "TrackGTime": track_time,
                "TrackID": track_id,
                "TrackParamID": param_id,
                "TrackValue": clean_text(value, 255),
            })
            track_id += 1
            fixed_counter += 1

        body_part_name = normalize_body_part_name(row["parte_del_cuerpo"])

        if body_part_name != "":
            if body_part_name not in BODY_PART_PARAM_MAP:
                raise KeyError(
                    f"No existe parte_del_cuerpo configurada en catálogo: "
                    f"{body_part_name}"
                )

            records.append({
                "TrackGID": track_gid,
                "TrackGClinicID": TRACK_GCLINIC_ID,
                "TrackGClientID": client_id,
                "TrackGDiaryID": track_diary,
                "TrackGDate": track_date,
                "TrackGTime": track_time,
                "TrackID": track_id,
                "TrackParamID": BODY_PART_PARAM_ID,
                "TrackValue": body_part_name,
            })
            track_id += 1
            body_part_counter += 1

        for col, param_id in MEASURE_PARAMS.items():
            value = row[col]

            if pd.isna(value) or str(value).strip() == "":
                continue

            records.append({
                "TrackGID": track_gid,
                "TrackGClinicID": TRACK_GCLINIC_ID,
                "TrackGClientID": client_id,
                "TrackGDiaryID": track_diary,
                "TrackGDate": track_date,
                "TrackGTime": track_time,
                "TrackID": track_id,
                "TrackParamID": param_id,
                "TrackValue": clean_text(value, 255),
            })
            track_id += 1
            measure_counter += 1

        track_gid += 1
        track_diary += 1

    df_dest = pd.DataFrame(records)

    for col in DEST_COLS:
        if col not in df_dest.columns:
            df_dest[col] = ""

    df_dest = df_dest[DEST_COLS]

    print(f"registros_generados_fecha_lectura = {date_counter}")
    print(f"registros_generados_parametros_fijos = {fixed_counter}")
    print(f"registros_generados_partes_del_cuerpo = {body_part_counter}")
    print(f"registros_generados_medidas_corporales = {measure_counter}")
    print(f"registros_finales_clients_tracking_gen = {len(df_dest)}")

    return df_dest


# =========================================================
# CONSTRUCCIÓN DESTINO DIARY
# =========================================================

def build_diary_hidden_df(df_tracking: pd.DataFrame) -> pd.DataFrame:
    """
    Crea un registro por cada TrackGID distinto.

    Reglas:
    - DiaryID debe coincidir con TrackGDiaryID
    - DiaryGID debe coincidir con TrackGDiaryID
    - ambos empiezan en 300 si TrackGDiaryID empieza en 300
    """
    df_diary_base = (
        df_tracking[["TrackGID", "TrackGDiaryID", "TrackGClientID", "TrackGDate", "TrackGTime"]]
        .copy()
        .rename(columns={
            "TrackGID": "SourceTrackGID",
            "TrackGDiaryID": "DiaryID",
            "TrackGClientID": "DiaryGClientID",
            "TrackGDate": "DiaryGDate",
            "TrackGTime": "BaseTrackGTime",
        })
    )

    df_diary_base["DiaryID"] = df_diary_base["DiaryID"].astype(str).str.strip()
    df_diary_base["DiaryGClientID"] = df_diary_base["DiaryGClientID"].astype(str).str.strip()
    df_diary_base["DiaryGDate"] = df_diary_base["DiaryGDate"].astype(str).str.strip()
    df_diary_base["BaseTrackGTime"] = df_diary_base["BaseTrackGTime"].astype(str).str.strip()

    df_diary_base = df_diary_base[
        df_diary_base["DiaryID"].ne("")
        & df_diary_base["DiaryGClientID"].ne("")
        & df_diary_base["DiaryGDate"].ne("")
    ].copy()

    df_diary_base = (
        df_diary_base
        .sort_values(
            ["SourceTrackGID", "DiaryID", "DiaryGClientID", "DiaryGDate", "BaseTrackGTime"],
            kind="mergesort"
        )
        .drop_duplicates(subset=["SourceTrackGID"], keep="first")
        .reset_index(drop=True)
    )

    df_diary_base["BaseTrackGTime"] = df_diary_base["BaseTrackGTime"].replace({"": np.nan})
    first_time_by_date = (
        df_diary_base
        .sort_values(["DiaryGDate", "BaseTrackGTime", "SourceTrackGID"], kind="mergesort")
        .groupby("DiaryGDate", as_index=False)
        .first()[["DiaryGDate", "BaseTrackGTime"]]
        .rename(columns={"BaseTrackGTime": "DiaryBaseStart"})
    )

    df_diary_base = df_diary_base.merge(first_time_by_date, on="DiaryGDate", how="left")

    df_diary_base = df_diary_base.sort_values(
        ["DiaryGDate", "SourceTrackGID"],
        kind="mergesort"
    ).reset_index(drop=True)

    df_diary_base["date_slot_order"] = df_diary_base.groupby("DiaryGDate").cumcount()

    df_diary_base["DiaryGStart"] = df_diary_base.apply(
        lambda r: add_minutes_hhmm(r["DiaryBaseStart"], int(r["date_slot_order"]) * 30),
        axis=1
    )
    df_diary_base["DiaryGEnd"] = df_diary_base["DiaryGStart"].apply(add_30_minutes_hhmm)

    df_diary_base["DiaryGClinicID"] = DIARY_GCLINIC_ID

    # Cambio solicitado:
    # DiaryGID debe ser el mismo autoincrement que TrackGDiaryID / DiaryID
    df_diary_base["DiaryGID"] = df_diary_base["DiaryID"]

    df_diary = df_diary_base[
        [
            "DiaryGID",
            "DiaryID",
            "DiaryGClientID",
            "DiaryGClinicID",
            "DiaryGDate",
            "DiaryGStart",
            "DiaryGEnd",
        ]
    ].copy()

    for col in DIARY_DEST_COLS:
        if col not in df_diary.columns:
            df_diary[col] = ""

    df_diary = df_diary[DIARY_DEST_COLS]

    print(f"registros_finales_diary_gen_hidden = {len(df_diary)}")

    return df_diary


# =========================================================
# MAIN
# =========================================================

def main():

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    engine = build_engine()

    logger.info("📥 Ejecutando consulta SQL...")

    df_src = pd.read_sql(SQL_QUERY, engine)

    print(f"total_filas_encontradas_en_sql_{SOURCE_TABLE_NAME} = {len(df_src)}")

    validate_source_columns(df_src)

    df_src = df_src[ORIG_COLS].copy()

    df_src["parte_del_cuerpo"] = df_src["parte_del_cuerpo"].apply(normalize_body_part_name)

    validate_body_part_mapping(df_src)

    df_src["_fecha_dt"] = safe_to_datetime(df_src["fecha"])

    body_part_unique = (
        df_src["parte_del_cuerpo"]
        .replace({"": np.nan})
        .dropna()
        .nunique()
    )
    print(f"valores_unicos_parte_del_cuerpo = {body_part_unique}")

    print("\n=== HEAD DF ORIGEN NORMALIZADO ===")
    print(df_src.head(20))

    df_dest = build_dest_df(df_src)

    df_diary_hidden = build_diary_hidden_df(df_dest)

    tracking_output_path = OUTPUT_DIR / TRACKING_OUTPUT_FILENAME
    df_dest.to_csv(
        tracking_output_path,
        sep=CSV_SEPARATOR,
        encoding=CSV_ENCODING,
        index=False,
        na_rep="\\N",
        lineterminator="\n",
    )

    print(f"total_final_clients_tracking_gen_csv = {len(df_dest)}")
    print(f"CSV generado en: {tracking_output_path}")

    diary_output_path = OUTPUT_DIR / DIARY_OUTPUT_FILENAME
    df_diary_hidden.to_csv(
        diary_output_path,
        sep=CSV_SEPARATOR,
        encoding=CSV_ENCODING,
        index=False,
        na_rep="\\N",
        lineterminator="\n",
    )

    print(f"total_final_diary_gen_hidden_csv = {len(df_diary_hidden)}")
    print(f"CSV generado en: {diary_output_path}")

    logger.info("✅ Proceso completado")


# =========================================================
# ENTRYPOINT
# =========================================================

if __name__ == "__main__":
    main()